# LeetCode 3216 — Lexicographically Smallest String After a Swap bằng Python

Link đề: https://leetcode.com/problems/lexicographically-smallest-string-after-a-swap/ · Độ khó: **Easy** · Chủ đề: Greedy, String

Skeleton LeetCode:

```python
class Solution:
    def getSmallestString(self, s: str) -> str:
        ...
```

Ghi chú: đề và các ví dụ được viết theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho chuỗi `s` chỉ gồm **chữ số**; được phép **hoán đổi hai chữ số đứng cạnh nhau cùng tính chẵn lẻ** (cùng chẵn hoặc cùng lẻ) **tối đa một lần**; trả về chuỗi **nhỏ nhất theo thứ tự từ điển** (lexicographically smallest) có thể có.
2. Nắm các công cụ Python: so sánh chuỗi bằng `<` theo thứ tự từ điển, chẵn lẻ của chữ số `int(c) % 2`, cắt lát và ghép chuỗi `s[:i] + s[i+1] + s[i] + s[i+2:]`, hoán đổi trong list `cs[i], cs[i+1] = cs[i+1], cs[i]`, `min(...)` trên các chuỗi, và `zip(s, s[1:])` / `enumerate` / `next(...)`.
3. Đi từ **thử mọi cách hoán đổi hợp lệ rồi chọn chuỗi nhỏ nhất** → **hoán đổi đầu tiên làm chuỗi nhỏ đi** → **tham lam: cặp trái-sang-phải đầu tiên cùng chẵn lẻ và `trái > phải`** → **bản dùng list hoán đổi tại chỗ** → **bản `next(...)` một dòng**.
4. Hiểu mấu chốt: thứ tự từ điển **ưu tiên vị trí bên trái**, nên hoán đổi **đẹp nhất là hoán đổi ở vị trí sớm nhất mà làm chuỗi nhỏ đi**; hoán đổi một lần xong là **dừng**.
5. Tránh các lỗi kinh điển: **bỏ qua điều kiện cùng chẵn lẻ**, dùng `>=` (hoán đổi hai chữ số bằng nhau làm phí lượt), **hoán đổi nhiều lần** (đề chỉ cho một lần), và chọn hoán đổi "giảm nhiều nhất" thay vì "**sớm nhất**".

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ chứa các **chữ số**. Hãy trả về **chuỗi nhỏ nhất theo thứ tự từ điển** có thể thu được sau khi **hoán đổi hai chữ số liền kề có cùng tính chẵn lẻ** (parity) **tối đa một lần**.

Hai chữ số **cùng tính chẵn lẻ** nếu cùng lẻ hoặc cùng chẵn. Ví dụ `5` và `9` cùng lẻ, `2` và `4` cùng chẵn, còn `6` và `9` thì **khác** tính chẵn lẻ.

**Ví dụ 1**

- Input: `s = "45320"`
- Output: `"43520"`
- Giải thích: `s[1] = '5'` và `s[2] = '3'` cùng lẻ; đổi chỗ hai chữ số này cho chuỗi nhỏ nhất.

**Ví dụ 2**

- Input: `s = "001"`
- Output: `"001"`
- Giải thích: không cần hoán đổi vì chuỗi đã là nhỏ nhất.

**Ràng buộc:** `2 <= s.length <= 100`; `s` chỉ gồm các chữ số.

In [1]:
from typing import List
import random
from itertools import product

examples = [
    ("45320", "43520"),
    ("001", "001"),
]
for s, expected in examples:
    print(f"s = {s!r} -> kỳ vọng {expected!r}")

s = '45320' -> kỳ vọng '43520'
s = '001' -> kỳ vọng '001'


## 3. Kiến thức nền

### 3.1. Thứ tự từ điển (lexicographic): so từ trái sang phải

Hai chuỗi được so **ký tự đầu tiên khác nhau từ trái sang phải**; chuỗi nào có ký tự nhỏ hơn tại đó thì nhỏ hơn, các ký tự phía sau **không quan trọng**. Python so sánh chuỗi bằng `<`, `>`, `==` theo đúng quy tắc này. Với chữ số `'0' < '1' < ... < '9'`, nên so chuỗi gồm các chữ số đúng như so từng chữ.

In [2]:
print("43520" < "45320")        # True: khác nhau đầu tiên ở vị trí 1, '3' < '5'
print("0999" < "1000")          # True: vị trí 0, '0' < '1', các vị trí sau không quan trọng
print("001" < "010")            # True
print("abc" < "abd", "abc" < "ab")    # True False  (chuỗi dài hơn mà có tiền tố giống thì lớn hơn)
print(min(["45320", "54320", "43520"]))     # 43520

True
True
True
True False
43520


### 3.2. Chẵn / lẻ của một chữ số

`int(c)` đổi ký tự chữ số thành số nguyên; `% 2` cho `0` (chẵn) hoặc `1` (lẻ). Hai chữ số **cùng chẵn lẻ** khi `int(a) % 2 == int(b) % 2`, tương đương `(int(a) + int(b)) % 2 == 0` (tổng chẵn). Mẹo thêm: vì `ord('0') = 48` là số chẵn nên `ord(c) % 2` cũng bằng `int(c) % 2`.

In [3]:
for a, b in [("5", "9"), ("2", "4"), ("6", "9"), ("0", "0"), ("7", "8")]:
    same = int(a) % 2 == int(b) % 2
    print(a, b, "cùng chẵn lẻ?" , same, "| tổng chẵn?", (int(a) + int(b)) % 2 == 0, "| ord:", ord(a) % 2 == ord(b) % 2)

5 9 cùng chẵn lẻ? True | tổng chẵn? True | ord: True
2 4 cùng chẵn lẻ? True | tổng chẵn? True | ord: True
6 9 cùng chẵn lẻ? False | tổng chẵn? False | ord: False
0 0 cùng chẵn lẻ? True | tổng chẵn? True | ord: True
7 8 cùng chẵn lẻ? False | tổng chẵn? False | ord: False


### 3.3. Hoán đổi hai ký tự liền kề: cắt lát hoặc list

Chuỗi là **bất biến** nên không gán `s[i] = ...` được. Hai cách hoán đổi `s[i]` và `s[i+1]`:

1. **Cắt lát và ghép:** `s[:i] + s[i+1] + s[i] + s[i+2:]` (phần đầu, rồi chữ thứ `i+1`, chữ thứ `i`, phần còn lại).
2. **Chuyển sang list:** `cs = list(s)`, hoán đổi `cs[i], cs[i+1] = cs[i+1], cs[i]`, rồi `"".join(cs)`.

In [4]:
s, i = "45320", 1
print(s[:i], s[i + 1], s[i], s[i + 2:])                  # 4 3 5 20
print(s[:i] + s[i + 1] + s[i] + s[i + 2:])               # 43520
cs = list(s)
cs[i], cs[i + 1] = cs[i + 1], cs[i]
print("".join(cs))                                       # 43520

4 3 5 20
43520
43520


### 3.4. Duyệt các cặp liền kề: `range(len(s) - 1)` và `zip(s, s[1:])`

Có `n` chữ số thì có `n - 1` cặp liền kề `(s[i], s[i+1])` với `i = 0..n-2`. Dùng `range(len(s) - 1)` để không đọc `s[n]` (vượt biên). Cách khác: `zip(s, s[1:])` ghép mỗi chữ với chữ đứng sau nó, tự dừng khi hết cặp.

In [5]:
s = "45320"
for i in range(len(s) - 1):
    print(i, s[i], "->", s[i + 1])
print(list(zip(s, s[1:])))
print([(i, a, b) for i, (a, b) in enumerate(zip(s, s[1:]))])

0 4 -> 5
1 5 -> 3
2 3 -> 2
3 2 -> 0
[('4', '5'), ('5', '3'), ('3', '2'), ('2', '0')]
[(0, '4', '5'), (1, '5', '3'), (2, '3', '2'), (3, '2', '0')]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `s = "45320"`, liệt kê **mọi** hoán đổi liền kề hợp lệ (cùng chẵn lẻ) và chuỗi nhận được:

| cặp (vị trí) | hai chữ số | cùng chẵn lẻ? | chuỗi sau hoán đổi |
|---|---|---|---|
| 0, 1 | 4, 5 | không (chẵn, lẻ) | — |
| 1, 2 | 5, 3 | **có** (cùng lẻ) | `43520` |
| 2, 3 | 3, 2 | không (lẻ, chẵn) | — |
| 3, 4 | 2, 0 | **có** (cùng chẵn) | `45302` |

Ngoài ra còn lựa chọn **không hoán đổi**: `45320`. So ba chuỗi `45320`, `43520`, `45302`: nhỏ nhất là `43520` (chữ số thứ hai `3` nhỏ hơn `5`). Nhìn kỹ: **hoán đổi ở vị trí 1** đưa chữ số **nhỏ hơn** (`3`) lên **trước** (vị trí 1), nên chuỗi giảm ngay từ vị trí 1. Hoán đổi ở vị trí 3 cũng đưa chữ số nhỏ hơn lên trước (`0` lên trước `2`), nhưng nó chỉ ảnh hưởng vị trí 3 — **bên phải**, nên kém hơn. Ý chính: **sớm nhất, nhỏ nhất**.

In [6]:
s = "45320"
candidates = {"(không đổi)": s}
for i in range(len(s) - 1):
    if int(s[i]) % 2 == int(s[i + 1]) % 2:
        candidates[f"đổi vị trí {i},{i + 1}"] = s[:i] + s[i + 1] + s[i] + s[i + 2:]
for name, cand in candidates.items():
    print(f"{name:<18} -> {cand}")
print("nhỏ nhất:", min(candidates.values()))

(không đổi)        -> 45320
đổi vị trí 1,2     -> 43520
đổi vị trí 3,4     -> 45302
nhỏ nhất: 43520


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán đổi hợp lệ, lấy chuỗi nhỏ nhất

**Ý tưởng.** Bắt đầu với `best = s` (không hoán đổi). Với mỗi `i` có `s[i]` và `s[i+1]` cùng chẵn lẻ, tạo chuỗi sau hoán đổi `cand`; nếu `cand < best` thì cập nhật `best`. Cuối cùng trả `best`. Làm đúng như đề, không cần nghĩ tới tham lam.

In [7]:
def smallest_bruteforce(s):
    best = s                                              # lựa chọn "không hoán đổi"
    for i in range(len(s) - 1):
        if int(s[i]) % 2 == int(s[i + 1]) % 2:            # cùng chẵn lẻ: được phép đổi
            cand = s[:i] + s[i + 1] + s[i] + s[i + 2:]
            if cand < best:
                best = cand
    return best


for s, expected in examples:
    print(repr(smallest_bruteforce(s)), "(kỳ vọng", repr(expected), ")")

'43520' (kỳ vọng '43520' )
'001' (kỳ vọng '001' )


**Độ phức tạp (Cách 1).** Gọi `n = len(s)`. Có tối đa `n - 1` hoán đổi; mỗi lần tạo chuỗi `cand` và so sánh tốn `O(n)` ⇒ **Time `O(n²)`**, **Space `O(n)`**. Với `n <= 100` thì tức thì.

**Nút thắt:** ta dựng và so sánh nhiều chuỗi dài, trong khi chỉ cần biết **vị trí hoán đổi nào đáng làm**. Đáng làm là hoán đổi làm chuỗi **nhỏ đi**, và ta chỉ cần cái **sớm nhất**.

## 6. Ý tưởng thuật toán / bước đột phá

Xét hoán đổi tại vị trí `i` (hai chữ số `a = s[i]`, `b = s[i+1]`, cùng chẵn lẻ). Chuỗi mới `cand` **chỉ khác `s` ở hai vị trí `i` và `i + 1`**:

- Nếu `a > b`: `cand` **nhỏ hơn** `s` (tại vị trí `i`, `b < a`).
- Nếu `a == b`: `cand` **bằng** `s` (hoán đổi vô nghĩa).
- Nếu `a < b`: `cand` **lớn hơn** `s` (hoán đổi làm xấu đi).

Vậy chỉ các hoán đổi có **`a > b` và cùng chẵn lẻ** đáng làm. Giữa hai hoán đổi như vậy ở `i < j`, `cand_i` và `cand_j` **giống nhau ở các vị trí `< i`**; tại vị trí `i` thì `cand_i[i] = b < a = s[i] = cand_j[i]` (hoán đổi `j` không đụng tới vị trí `i`, kể cả khi `j = i + 1`, vì nó chỉ đổi vị trí `i + 1` và `i + 2`). Do đó `cand_i < cand_j`: **vị trí càng trái càng tốt**.

**Thuật toán tham lam:** duyệt `i` từ trái sang phải, gặp cặp **đầu tiên** cùng chẵn lẻ và `s[i] > s[i+1]` thì hoán đổi và **trả về ngay**. Nếu không có cặp nào thì chuỗi đã nhỏ nhất, trả `s`.

In [8]:
s = "45320"
for i in range(len(s) - 1):
    a, b = int(s[i]), int(s[i + 1])
    same = a % 2 == b % 2
    print(f"i={i}: ({a}, {b}) cùng chẵn lẻ? {same}, {a} > {b}? {a > b}")
    if same and a > b:
        print("  -> hoán đổi ở đây và dừng:", s[:i] + s[i + 1] + s[i] + s[i + 2:])
        break

i=0: (4, 5) cùng chẵn lẻ? False, 4 > 5? False
i=1: (5, 3) cùng chẵn lẻ? True, 5 > 3? True
  -> hoán đổi ở đây và dừng: 43520


## 7. Các cách tiếp theo

### Cách 2 — Hoán đổi hợp lệ đầu tiên làm chuỗi nhỏ đi (so sánh chuỗi)

**Giải thích từng đoạn.**

1. Duyệt `i` từ trái sang phải; nếu `s[i]`, `s[i+1]` cùng chẵn lẻ thì tạo `cand`.
2. **Nếu `cand < s`** thì đây là hoán đổi sớm nhất làm chuỗi nhỏ đi: trả về `cand` ngay.
3. Hết vòng lặp không có hoán đổi nào làm nhỏ đi thì trả `s`.

Ưu điểm: không cần nghĩ tới điều kiện `a > b`, để Python so sánh chuỗi. Nhược điểm: mỗi bước dựng và so sánh chuỗi `O(n)`.

In [9]:
def smallest_first_improving(s):
    for i in range(len(s) - 1):
        if int(s[i]) % 2 == int(s[i + 1]) % 2:
            cand = s[:i] + s[i + 1] + s[i] + s[i + 2:]
            if cand < s:                      # hoán đổi đầu tiên làm chuỗi nhỏ đi
                return cand
    return s


for s, expected in examples:
    print(repr(smallest_first_improving(s)), "(kỳ vọng", repr(expected), ")")

'43520' (kỳ vọng '43520' )
'001' (kỳ vọng '001' )


In [10]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionFirstImproving:
    def getSmallestString(self, s: str) -> str:
        for i in range(len(s) - 1):
            if int(s[i]) % 2 == int(s[i + 1]) % 2:
                cand = s[:i] + s[i + 1] + s[i] + s[i + 2:]
                if cand < s:
                    return cand
        return s


print(SolutionFirstImproving().getSmallestString("45320"))      # 43520

43520


**Độ phức tạp Cách 2:** **Time `O(n²)`** (dựng và so sánh chuỗi mỗi bước), **Space `O(n)`**.

### Cách 3 — Tham lam: cặp đầu tiên cùng chẵn lẻ và `trái > phải` (cách nên nộp)

**Giải thích từng đoạn.**

1. `a, b = int(s[i]), int(s[i+1])` đổi hai chữ số sang số nguyên.
2. `a % 2 == b % 2 and a > b`: cùng chẵn lẻ **và** chữ số trái lớn hơn chữ số phải.
3. Gặp cặp đầu tiên thỏa thì `return s[:i] + s[i+1] + s[i] + s[i+2:]` (hoán đổi đúng một lần rồi dừng).
4. Không có cặp nào: `return s`.

Chỉ **một lần quét**, mỗi bước so sánh hai chữ số `O(1)`, hoán đổi dựng chuỗi một lần ⇒ tổng `O(n)`.

In [11]:
def smallest_greedy(s):
    for i in range(len(s) - 1):
        a, b = int(s[i]), int(s[i + 1])
        if a % 2 == b % 2 and a > b:                     # cùng chẵn lẻ và trái > phải
            return s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s


for s, expected in examples:
    print(repr(smallest_greedy(s)), "(kỳ vọng", repr(expected), ")")

'43520' (kỳ vọng '43520' )
'001' (kỳ vọng '001' )


In [12]:
# Code nộp LeetCode cho Cách 3
class SolutionGreedy:
    def getSmallestString(self, s: str) -> str:
        for i in range(len(s) - 1):
            a, b = int(s[i]), int(s[i + 1])
            if a % 2 == b % 2 and a > b:
                return s[:i] + s[i + 1] + s[i] + s[i + 2:]
        return s


print(SolutionGreedy().getSmallestString("001"))      # 001

001


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (chuỗi kết quả; quét thì `O(1)`).

### Cách 4 — List + hoán đổi tại chỗ

**Giải thích từng đoạn.**

1. `cs = list(s)`: chuyển chuỗi bất biến sang list có thể sửa.
2. Tìm cặp đầu tiên `cs[i] > cs[i+1]` (so sánh ký tự chữ số đúng như so số) và `(ord(cs[i]) - ord(cs[i+1])) % 2 == 0` (hiệu chẵn ⇔ cùng chẵn lẻ).
3. Hoán đổi `cs[i], cs[i+1] = cs[i+1], cs[i]` rồi `break` (**phải dừng**, nếu không sẽ hoán đổi nhiều lần).
4. `"".join(cs)`.

Cách này tránh cắt lát và dễ mở rộng nếu sau này cần sửa nhiều chỗ.

In [13]:
def smallest_list_swap(s):
    cs = list(s)
    for i in range(len(cs) - 1):
        if cs[i] > cs[i + 1] and (ord(cs[i]) - ord(cs[i + 1])) % 2 == 0:
            cs[i], cs[i + 1] = cs[i + 1], cs[i]
            break                                   # chỉ một lần hoán đổi
    return "".join(cs)


for s, expected in examples:
    print(repr(smallest_list_swap(s)), "(kỳ vọng", repr(expected), ")")

'43520' (kỳ vọng '43520' )
'001' (kỳ vọng '001' )


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionListSwap:
    def getSmallestString(self, s: str) -> str:
        cs = list(s)
        for i in range(len(cs) - 1):
            if cs[i] > cs[i + 1] and (ord(cs[i]) - ord(cs[i + 1])) % 2 == 0:
                cs[i], cs[i + 1] = cs[i + 1], cs[i]
                break
        return "".join(cs)


print(SolutionListSwap().getSmallestString("45320"))      # 43520

43520


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`**.

### Cách 5 — Tìm chỉ số bằng `next(...)` trên `zip`

**Giải thích từng đoạn.**

1. `enumerate(zip(s, s[1:]))` sinh `(i, (a, b))` cho từng cặp liền kề.
2. `next(generator, -1)` lấy chỉ số `i` **đầu tiên** thỏa điều kiện, hoặc `-1` nếu không có.
3. Nếu `i == -1` trả `s`; ngược lại hoán đổi hai chữ số ở `i`.

Cách này tách "tìm vị trí" khỏi "thực hiện hoán đổi".

In [15]:
def smallest_next(s):
    i = next((i for i, (a, b) in enumerate(zip(s, s[1:]))
              if a > b and (int(a) + int(b)) % 2 == 0), -1)
    if i == -1:
        return s
    return s[:i] + s[i + 1] + s[i] + s[i + 2:]


for s, expected in examples:
    print(repr(smallest_next(s)), "(kỳ vọng", repr(expected), ")")

'43520' (kỳ vọng '43520' )
'001' (kỳ vọng '001' )


In [16]:
# Code nộp LeetCode cho Cách 5
class SolutionNext:
    def getSmallestString(self, s: str) -> str:
        i = next((i for i, (a, b) in enumerate(zip(s, s[1:]))
                  if a > b and (int(a) + int(b)) % 2 == 0), -1)
        if i == -1:
            return s
        return s[:i] + s[i + 1] + s[i] + s[i + 2:]


print(SolutionNext().getSmallestString("45320"))      # 43520

43520


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`**.

## 8. TRACE từng bước

Ta trace **Cách 3** (tham lam) trên `s = "45320"`, rồi trên `"2468"` (không có hoán đổi nào) và `"5391"` (nhiều ứng viên).

Bảng cho `s = "45320"`:

| `i` | `a` | `b` | `a % 2 == b % 2`? | `a > b`? | hành động |
|---|---|---|---|---|---|
| 0 | 4 | 5 | không (chẵn, lẻ) | không | tiếp tục |
| 1 | 5 | 3 | **có** (cùng lẻ) | **có** | hoán đổi, **trả `"43520"`** |

(Các vị trí `i = 2, 3` không cần xét vì đã dừng.)

Bảng cho `s = "2468"`: `i=0`: `(2,4)` cùng chẵn, `2 > 4`? không; `i=1`: `(4,6)` cùng chẵn, `4 > 6`? không; `i=2`: `(6,8)` cùng chẵn, `6 > 8`? không; hết vòng lặp ⇒ trả `"2468"` (đã tăng dần nên đã nhỏ nhất).

Với `s = "5391"`: `i=0`: `(5,3)` cùng lẻ, `5 > 3` ⇒ hoán đổi, trả `"3591"`; dù ở `i=2` có `(9,1)` cùng lẻ và `9 > 1` (chênh lớn hơn) nhưng ta không xét vì đã dừng — và `"3591" < "5319"` nên chọn hoán đổi sớm hơn là đúng.

In [17]:
def trace_greedy(s):
    print(f"s = {s!r}")
    for i in range(len(s) - 1):
        a, b = int(s[i]), int(s[i + 1])
        same = a % 2 == b % 2
        print(f"  i={i}: a={a}, b={b}, cùng chẵn lẻ? {same}, a > b? {a > b}")
        if same and a > b:
            result = s[:i] + s[i + 1] + s[i] + s[i + 2:]
            print(f"  -> hoán đổi tại i={i}: {s[:i]!r} + {s[i + 1]!r} + {s[i]!r} + {s[i + 2:]!r} = {result!r}")
            return result
    print("  -> không có cặp nào thỏa, giữ nguyên")
    return s


assert trace_greedy("45320") == "43520"
assert trace_greedy("2468") == "2468"
assert trace_greedy("5391") == "3591"

s = '45320'
  i=0: a=4, b=5, cùng chẵn lẻ? False, a > b? False
  i=1: a=5, b=3, cùng chẵn lẻ? True, a > b? True
  -> hoán đổi tại i=1: '4' + '3' + '5' + '20' = '43520'
s = '2468'
  i=0: a=2, b=4, cùng chẵn lẻ? True, a > b? False
  i=1: a=4, b=6, cùng chẵn lẻ? True, a > b? False
  i=2: a=6, b=8, cùng chẵn lẻ? True, a > b? False
  -> không có cặp nào thỏa, giữ nguyên
s = '5391'
  i=0: a=5, b=3, cùng chẵn lẻ? True, a > b? True
  -> hoán đổi tại i=0: '' + '3' + '5' + '91' = '3591'


In [18]:
def trace_bruteforce(s):
    print(f"s = {s!r}")
    best = s
    print(f"  bắt đầu best = {best!r} (không hoán đổi)")
    for i in range(len(s) - 1):
        if int(s[i]) % 2 == int(s[i + 1]) % 2:
            cand = s[:i] + s[i + 1] + s[i] + s[i + 2:]
            better = cand < best
            print(f"  i={i}: cùng chẵn lẻ, cand={cand!r}, cand < best? {better}")
            if better:
                best = cand
        else:
            print(f"  i={i}: khác chẵn lẻ, bỏ qua")
    print("  -> kết quả", repr(best))
    return best


assert trace_bruteforce("45320") == "43520"

s = '45320'
  bắt đầu best = '45320' (không hoán đổi)
  i=0: khác chẵn lẻ, bỏ qua
  i=1: cùng chẵn lẻ, cand='43520', cand < best? True
  i=2: khác chẵn lẻ, bỏ qua
  i=3: cùng chẵn lẻ, cand='45302', cand < best? False
  -> kết quả '43520'


## 9. Code nộp LeetCode

Bản nộp dùng **Cách 3** (tham lam, một lượt quét, hoán đổi đúng một lần rồi trả về).

In [19]:
class Solution:
    def getSmallestString(self, s: str) -> str:
        for i in range(len(s) - 1):
            a, b = int(s[i]), int(s[i + 1])
            if a % 2 == b % 2 and a > b:                         # cùng chẵn lẻ, trái lớn hơn phải
                return s[:i] + s[i + 1] + s[i] + s[i + 2:]       # hoán đổi đúng một lần rồi dừng
        return s                                                  # không có hoán đổi nào làm nhỏ đi


sol = Solution()
for s, expected in examples:
    print(repr(s), "->", repr(sol.getSmallestString(s)), "(kỳ vọng", repr(expected), ")")

'45320' -> '43520' (kỳ vọng '43520' )
'001' -> '001' (kỳ vọng '001' )


## 10. Vì sao thuật toán đúng

- **Mỗi hoán đổi hợp lệ chỉ đổi hai vị trí.** Với `s` và hoán đổi tại `i` (hai chữ số `a = s[i]`, `b = s[i+1]` cùng chẵn lẻ), chuỗi mới `cand` giống `s` ở mọi vị trí khác `i`, `i + 1`. Nên so `cand` với `s` chỉ cần xét vị trí `i`: `b < a` ⇒ `cand < s`; `b = a` ⇒ `cand = s`; `b > a` ⇒ `cand > s`.
- **Hoán đổi làm chuỗi nhỏ đi ⇔ `a > b`.** Vậy ứng viên tốt chỉ là các `i` có `a > b` và cùng chẵn lẻ. Nếu không có `i` nào, mọi hoán đổi đều làm chuỗi lớn lên hoặc giữ nguyên, nên **không hoán đổi** là tốt nhất: trả `s`.
- **Sớm hơn thì nhỏ hơn.** Với hai ứng viên `i < j`: hai chuỗi giống nhau ở các vị trí `< i`; tại vị trí `i`, `cand_i[i] = s[i+1] < s[i] = cand_j[i]` (hoán đổi `j` không chạm vị trí `i` vì `j >= i + 1` chỉ đổi các vị trí `j, j + 1 >= i + 1`). Do thứ tự từ điển ưu tiên vị trí đầu tiên khác nhau, `cand_i < cand_j`.
- **Chỉ một lần.** Đề cho **tối đa một lần**; sau khi hoán đổi ta trả về ngay, không tiếp tục.
- **Kết luận:** hoán đổi tại cặp **đầu tiên** (trái sang phải) cùng chẵn lẻ có `a > b` cho chuỗi nhỏ nhất; không có cặp đó thì trả `s`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (`2 <= n <= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi hoán đổi, lấy `min` | `O(n²)` | `O(n)` |
| 2. Hoán đổi đầu tiên làm chuỗi nhỏ đi | `O(n²)` | `O(n)` |
| 3. Tham lam `a % 2 == b % 2 and a > b` | `O(n)` | `O(n)` |
| 4. List + hoán đổi tại chỗ | `O(n)` | `O(n)` |
| 5. `next` trên `zip` | `O(n)` | `O(n)` |

Time `O(n)` của Cách 3: quét tối đa `n - 1` cặp (mỗi cặp `O(1)`) rồi dựng một chuỗi kết quả `O(n)`. Space `O(n)` là do chuỗi kết quả (phần quét chỉ cần `O(1)`).

## 12. Bộ test

Ta gom mọi cách vào `impls` rồi so với kỳ vọng.

In [20]:
impls = {
    "bruteforce": smallest_bruteforce,
    "first_improving": smallest_first_improving,
    "greedy": smallest_greedy,
    "list_swap": smallest_list_swap,
    "next": smallest_next,
    "leetcode": Solution().getSmallestString,
    "leetcode_c2": SolutionFirstImproving().getSmallestString,
    "leetcode_c3": SolutionGreedy().getSmallestString,
    "leetcode_c4": SolutionListSwap().getSmallestString,
    "leetcode_c5": SolutionNext().getSmallestString,
}


def check_all(s, expected, label):
    for key, fn in impls.items():
        actual = fn(s)
        assert actual == expected, (label, key, s, actual, expected)
    print("PASS", label)


for idx, (s, expected) in enumerate(examples, 1):
    check_all(s, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

In [21]:
check_all("00", "00", "hai chữ số giống nhau")
check_all("21", "21", "khác chẵn lẻ: không được đổi")
check_all("31", "13", "cùng lẻ, trái > phải")
check_all("13", "13", "cùng lẻ nhưng trái < phải: giữ nguyên")
check_all("42", "24", "cùng chẵn, trái > phải")
check_all("24", "24", "cùng chẵn, đã tăng dần")
check_all("2468", "2468", "toàn chẵn tăng dần")
check_all("9731", "7931", "chỉ hoán đổi một lần, không sắp xếp hết")
check_all("5391", "3591", "hoán đổi sớm nhất (chênh nhỏ) tốt hơn hoán đổi muộn (chênh lớn)")
check_all("5531", "5351", "bỏ qua cặp bằng nhau, đổi cặp (5, 3) tiếp theo")
check_all("4310", "4130", "cặp (4, 3) khác chẵn lẻ nên bị bỏ qua; đổi cặp (3, 1)")
check_all("2864", "2684", "bài tính tay trong mục 17")
check_all("6572", "6572", "không có cặp cùng chẵn lẻ nào giảm")
check_all("8421", "4821", "(8, 4) cùng chẵn giảm: đổi ngay")
check_all("9" * 100, "9" * 100, "100 chữ số giống nhau")
rep = "1357" * 25
check_all(rep, rep[:3] + "1" + "7" + rep[5:], "1357 lặp lại: cặp (7, 1) ở chỗ nối là cặp giảm cùng chẵn lẻ đầu tiên")
cyc = "0123456789" * 10
check_all(cyc, cyc, "chữ số liền nhau luôn khác chẵn lẻ (kể cả 9 rồi 0): không có hoán đổi nào")

PASS hai chữ số giống nhau
PASS khác chẵn lẻ: không được đổi
PASS cùng lẻ, trái > phải
PASS cùng lẻ nhưng trái < phải: giữ nguyên
PASS cùng chẵn, trái > phải
PASS cùng chẵn, đã tăng dần
PASS toàn chẵn tăng dần
PASS chỉ hoán đổi một lần, không sắp xếp hết
PASS hoán đổi sớm nhất (chênh nhỏ) tốt hơn hoán đổi muộn (chênh lớn)
PASS bỏ qua cặp bằng nhau, đổi cặp (5, 3) tiếp theo
PASS cặp (4, 3) khác chẵn lẻ nên bị bỏ qua; đổi cặp (3, 1)
PASS bài tính tay trong mục 17
PASS không có cặp cùng chẵn lẻ nào giảm
PASS (8, 4) cùng chẵn giảm: đổi ngay
PASS 100 chữ số giống nhau
PASS 1357 lặp lại: cặp (7, 1) ở chỗ nối là cặp giảm cùng chẵn lẻ đầu tiên
PASS chữ số liền nhau luôn khác chẵn lẻ (kể cả 9 rồi 0): không có hoán đổi nào


### 12.2. Phản ví dụ cho các cách làm sai

Bốn lỗi phổ biến (chi tiết ở mục 13): (a) bỏ điều kiện cùng chẵn lẻ; (b) dùng `>=` thay vì `>`; (c) hoán đổi nhiều lần; (d) chọn hoán đổi "giảm nhiều nhất" thay vì "sớm nhất".

In [22]:
def wrong_no_parity(s):
    for i in range(len(s) - 1):
        if s[i] > s[i + 1]:                                              # SAI: thiếu điều kiện cùng chẵn lẻ
            return s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s


def wrong_ge(s):
    for i in range(len(s) - 1):
        a, b = int(s[i]), int(s[i + 1])
        if a % 2 == b % 2 and a >= b:                                    # SAI: >= tính cả hai chữ số bằng nhau
            return s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s


def wrong_many_swaps(s):
    cs = list(s)
    for i in range(len(cs) - 1):
        if cs[i] > cs[i + 1] and (int(cs[i]) + int(cs[i + 1])) % 2 == 0:
            cs[i], cs[i + 1] = cs[i + 1], cs[i]                          # SAI: không break, hoán đổi nhiều lần
    return "".join(cs)


def wrong_max_drop(s):
    best_i, best_drop = -1, 0
    for i in range(len(s) - 1):
        a, b = int(s[i]), int(s[i + 1])
        if a % 2 == b % 2 and a - b > best_drop:                         # SAI: chọn cặp chênh lệch lớn nhất
            best_i, best_drop = i, a - b
    if best_i == -1:
        return s
    return s[:best_i] + s[best_i + 1] + s[best_i] + s[best_i + 2:]


print(wrong_no_parity("4310"), "!=", Solution().getSmallestString("4310"))
print(wrong_ge("5531"), "!=", Solution().getSmallestString("5531"))
print(wrong_many_swaps("9731"), "!=", Solution().getSmallestString("9731"))
print(wrong_max_drop("5391"), "!=", Solution().getSmallestString("5391"))
assert wrong_no_parity("4310") != "4130"
assert wrong_ge("5531") != "5351"
assert wrong_many_swaps("9731") != "7931"
assert wrong_max_drop("5391") != "3591"

3410 != 4130
5531 != 5351
7319 != 7931
5319 != 3591


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: chuỗi chữ số độ dài `2..100`. Ta kiểm tra: (1) **ngẫu nhiên** `3000` chuỗi với seed cố định so `10` cài đặt; (2) **vét cạn** mọi chuỗi chữ số độ dài `2, 3, 4` (`11100` chuỗi); (3) tính chất: kết quả **không lớn hơn** `s`, là **hoán vị** của `s`, và **khác `s` tối đa ở hai vị trí liền kề**.

In [23]:
rng = random.Random(3216)
for trial in range(3000):
    n_len = rng.choice([2, 3, 4, 5, 8, 15, 30, 60, 100])
    alphabet = rng.choice(["0123456789", "02468", "13579", "0123", "5391", "01"])
    s = "".join(rng.choice(alphabet) for _ in range(n_len))
    ref = smallest_bruteforce(s)
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s)
    assert ref <= s and sorted(ref) == sorted(s)
    diff = [i for i in range(n_len) if ref[i] != s[i]]
    assert diff == [] or (len(diff) == 2 and diff[1] == diff[0] + 1)
print("PASS 3000 chuỗi ngẫu nhiên (kèm kiểm tra tính chất)")

total = 0
for n_len in range(2, 5):
    for digits in product("0123456789", repeat=n_len):
        s = "".join(digits)
        ref = smallest_bruteforce(s)
        assert smallest_first_improving(s) == ref
        assert smallest_greedy(s) == ref
        assert smallest_list_swap(s) == ref
        assert smallest_next(s) == ref
        assert Solution().getSmallestString(s) == ref
        total += 1
print("PASS vét cạn", total, "chuỗi chữ số độ dài 2..4")

PASS 3000 chuỗi ngẫu nhiên (kèm kiểm tra tính chất)


PASS vét cạn 11100 chuỗi chữ số độ dài 2..4


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Bỏ qua điều kiện "cùng chẵn lẻ"

In [24]:
def sai_khong_chan_le(s):
    for i in range(len(s) - 1):
        if s[i] > s[i + 1]:                       # SAI: gặp cặp giảm đầu tiên là đổi, không xét chẵn lẻ
            return s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s


print("sai:", sai_khong_chan_le("4310"), "| đúng:", Solution().getSmallestString("4310"))
print("sai:", sai_khong_chan_le("21"), "| đúng:", Solution().getSmallestString("21"))

sai: 3410 | đúng: 4130
sai: 12 | đúng: 21


**Vì sao sai:** `4` và `3` khác chẵn lẻ nên đề **không cho** hoán đổi chúng. Code sai đổi thành `"3410"` — một chuỗi **không thể** đạt được theo luật. Phải kiểm tra cặp cùng chẵn lẻ **trước**, và với `"4310"` hoán đổi hợp lệ đầu tiên là `(3, 1)` cho `"4130"`. **Cách sửa:** thêm `int(a) % 2 == int(b) % 2` (hoặc `(int(a) + int(b)) % 2 == 0`).

### Lỗi 2: Dùng `>=` thay vì `>` (đổi hai chữ số bằng nhau)

In [25]:
def sai_lon_hon_hoac_bang(s):
    for i in range(len(s) - 1):
        a, b = int(s[i]), int(s[i + 1])
        if a % 2 == b % 2 and a >= b:             # SAI: a == b cũng được tính là "giảm"
            return s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s


print("sai:", sai_lon_hon_hoac_bang("5531"), "| đúng:", Solution().getSmallestString("5531"))

sai: 5531 | đúng: 5351


**Vì sao sai:** với `"5531"`, cặp `(5, 5)` bằng nhau; "hoán đổi" chúng **không đổi gì** nhưng code sai coi đó là hoán đổi duy nhất rồi **dừng**, bỏ lỡ cặp `(5, 3)` ở sau cho `"5351"`. **Cách sửa:** chỉ hoán đổi khi `a > b` **ngặt** (`a == b` bỏ qua và đi tiếp).

### Lỗi 3: Hoán đổi nhiều lần (không dừng sau lần đầu)

In [26]:
def sai_nhieu_lan(s):
    cs = list(s)
    for i in range(len(cs) - 1):
        if cs[i] > cs[i + 1] and (int(cs[i]) + int(cs[i + 1])) % 2 == 0:
            cs[i], cs[i + 1] = cs[i + 1], cs[i]       # SAI: thiếu break
    return "".join(cs)


print("sai:", sai_nhieu_lan("9731"), "| đúng:", Solution().getSmallestString("9731"))

sai: 7319 | đúng: 7931


**Vì sao sai:** đề cho **tối đa một lần** hoán đổi. Code sai "đẩy" chữ số `9` qua cả `7`, `3`, `1` (ba lần đổi) cho `"7319"`; thực tế chỉ được đổi một lần: `"7931"`. **Cách sửa:** `return` (hoặc `break`) ngay sau lần hoán đổi đầu tiên.

### Lỗi 4: Chọn hoán đổi "giảm nhiều nhất" thay vì "sớm nhất"

In [27]:
def sai_giam_nhieu_nhat(s):
    best_i, best_drop = -1, 0
    for i in range(len(s) - 1):
        a, b = int(s[i]), int(s[i + 1])
        if a % 2 == b % 2 and a - b > best_drop:
            best_i, best_drop = i, a - b
    if best_i == -1:
        return s
    return s[:best_i] + s[best_i + 1] + s[best_i] + s[best_i + 2:]


print("sai:", sai_giam_nhieu_nhat("5391"), "| đúng:", Solution().getSmallestString("5391"))

sai: 5319 | đúng: 3591


**Vì sao sai:** thứ tự từ điển ưu tiên **vị trí bên trái**, không ưu tiên độ lớn mức giảm. Với `"5391"`: đổi `(9, 1)` giảm tới `8` nhưng chuỗi `"5319"` bắt đầu bằng `5`; đổi `(5, 3)` chỉ giảm `2` nhưng cho `"3591"` bắt đầu bằng `3` nhỏ hơn. **Cách sửa:** lấy cặp **đầu tiên** trái sang phải thỏa điều kiện, không so sánh mức giảm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuỗi nhỏ nhất theo thứ tự từ điển với **ít phép đổi**" ⇒ **tham lam từ trái sang phải**: sửa ở vị trí **sớm nhất** có thể làm nhỏ đi.
- Hoán đổi hai phần tử liền kề: chỉ ảnh hưởng hai vị trí ⇒ so sánh chỉ cần hai phần tử đó (`a > b` ⇒ nhỏ đi).
- Điều kiện **cùng chẵn lẻ**: `a % 2 == b % 2` hoặc `(a + b) % 2 == 0`; với ký tự chữ số `ord(c) % 2` cũng dùng được (vì `ord('0') = 48` là số chẵn).
- Đề nói "**tối đa một lần**": nhớ `return`/`break` ngay sau lần đổi đầu tiên.
- Bỏ qua cặp **bằng nhau** (`a == b`): đổi không có tác dụng nhưng không được tính là "đã đổi".
- Chữ số liền nhau khác chẵn lẻ **chặn** nhau: một chữ số không thể "nhảy qua" chữ số khác chẵn lẻ khi chỉ được đổi liền kề.
- Luôn kiểm bằng **brute force** trên chuỗi ngắn (vét cạn mọi chuỗi chữ số độ dài `<= 4`).

## 15. Biến thể của bài

1. **Lớn nhất** thay vì nhỏ nhất: cặp đầu tiên cùng chẵn lẻ có `trái < phải`.
2. **Không giới hạn số lần** hoán đổi liền kề cùng chẵn lẻ: mỗi **đoạn liên tiếp cùng chẵn lẻ** được sắp xếp tăng dần.
3. **Hoán đổi bất kỳ hai chữ số** cùng chẵn lẻ (không cần liền kề) tối đa một lần.
4. **Tối đa `k` lần** hoán đổi liền kề (bỏ điều kiện chẵn lẻ): tham lam kiểu "đẩy chữ số nhỏ nhất về trước".
5. Bài liên quan: **Maximum Swap (670)**, **Minimum Adjacent Swaps to Reach the Kth Smallest Number (1850)**, **Smallest String With A Given Numeric Value (1663)**, **Maximum Odd Binary Number (2864)**.

In [28]:
def largest_one_swap(s):
    for i in range(len(s) - 1):
        a, b = int(s[i]), int(s[i + 1])
        if a % 2 == b % 2 and a < b:
            return s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s


def smallest_unlimited(s):
    """Không giới hạn số lần đổi: sắp xếp từng đoạn liên tiếp cùng chẵn lẻ."""
    out, i, n = [], 0, len(s)
    while i < n:
        j = i
        while j + 1 < n and int(s[j + 1]) % 2 == int(s[i]) % 2:
            j += 1
        out.append("".join(sorted(s[i:j + 1])))
        i = j + 1
    return "".join(out)


def smallest_unlimited_bfs(s):
    """Vét cạn (BFS) để kiểm tra smallest_unlimited trên chuỗi ngắn."""
    seen, frontier = {s}, [s]
    while frontier:
        nxt = []
        for cur in frontier:
            for i in range(len(cur) - 1):
                if int(cur[i]) % 2 == int(cur[i + 1]) % 2:
                    t = cur[:i] + cur[i + 1] + cur[i] + cur[i + 2:]
                    if t not in seen:
                        seen.add(t)
                        nxt.append(t)
        frontier = nxt
    return min(seen)


print(largest_one_swap("45320"), largest_one_swap("2468"))        # 45320 4268
print(smallest_unlimited("9731"), smallest_unlimited("45320"))      # 1379 43502
rng2 = random.Random(15)
for _ in range(200):
    t = "".join(rng2.choice("0123456789") for _ in range(rng2.randint(2, 6)))
    assert smallest_unlimited(t) == smallest_unlimited_bfs(t), t
print("smallest_unlimited khớp BFS trên 200 chuỗi ngắn")

45320 4268
1379 43502
smallest_unlimited khớp BFS trên 200 chuỗi ngắn


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi hoán đổi, lấy `min` | liệt kê hết ứng viên | `O(n²)` | `O(n)` | hiểu đề, kiểm chứng |
| 2. Hoán đổi đầu tiên làm chuỗi nhỏ đi | so `cand < s` | `O(n²)` | `O(n)` | khi muốn để Python so chuỗi |
| 3. Tham lam `a > b` cùng chẵn lẻ | cặp trái nhất | `O(n)` | `O(n)` | **nên nộp** |
| 4. List + hoán đổi tại chỗ | `cs[i], cs[i+1] = ...` rồi `break` | `O(n)` | `O(n)` | quen xử lý mảng tại chỗ |
| 5. `next` trên `zip` | tìm chỉ số rồi đổi | `O(n)` | `O(n)` | code gọn, tách tìm kiếm / thực hiện |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"43520" < "45320"`, `int("8") % 2 == int("2") % 2`, `"abc"[1] + "abc"[0]`, `Solution().getSmallestString("21")`, `Solution().getSmallestString("31")`.

**Bài 2 (tính tay).** Tìm đáp án cho ba chuỗi `"2864"`, `"6572"`, `"8421"`: với mỗi chuỗi lập bảng các cặp `(a, b)`, cùng chẵn lẻ không, `a > b` không, và chuỗi kết quả.

**Bài 3 (điền TODO).** Hoàn thành `get_smallest_todo(s)` theo Cách 3.

In [29]:
def get_smallest_todo(s):
    # TODO 1: duyệt i từ 0 đến len(s) - 2
    # TODO 2: a, b = int(s[i]), int(s[i + 1])
    # TODO 3: nếu a % 2 == b % 2 và a > b thì trả về s[:i] + s[i + 1] + s[i] + s[i + 2:]
    # TODO 4: sau vòng lặp (không có cặp nào) trả về s
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `smallest_bruteforce_todo(s)`: tạo **danh sách mọi chuỗi có thể đạt được** (kể cả không đổi) rồi trả `min(...)`; kiểm tra bằng cách so với `Solution` trên nhiều chuỗi ngắn.

**Bài 5 (biến thể / mini challenge).** (a) Viết `largest_todo(s)`: chuỗi **lớn nhất** sau tối đa một lần hoán đổi cùng chẵn lẻ liền kề. (b) Viết `smallest_unlimited_todo(s)`: nếu được hoán đổi **không giới hạn số lần**, chuỗi nhỏ nhất là gì? (gợi ý: các chữ số khác chẵn lẻ liền nhau là "bức tường" chia chuỗi thành các đoạn).

In [30]:
def smallest_bruteforce_todo(s):
    # TODO: candidates = [s] + [chuỗi sau khi đổi i, i+1 nếu cùng chẵn lẻ]; return min(candidates)
    raise NotImplementedError


def largest_todo(s):
    # TODO: tương tự Cách 3 nhưng tìm cặp cùng chẵn lẻ đầu tiên có a < b
    raise NotImplementedError


def smallest_unlimited_todo(s):
    # TODO: tách thành các đoạn cùng chẵn lẻ liên tiếp, sắp xếp từng đoạn tăng dần, ghép lại
    raise NotImplementedError

# KHÔNG gọi các hàm TODO này khi Run All.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True`, `True`, `'ba'`, `'21'` (hai chữ số khác chẵn lẻ), `'13'`. Bài 3: cặp đầu tiên `(a, b)` thỏa `a % 2 == b % 2 and a > b`.
- **Mức 2 (vừa):** Bài 2: `"2864"`: `(2,8)` cùng chẵn nhưng `2 < 8`; `(8,6)` cùng chẵn và `8 > 6` ⇒ `"2684"`. `"6572"`: `(6,5)` khác; `(5,7)` cùng lẻ nhưng `5 < 7`; `(7,2)` khác ⇒ giữ `"6572"`. `"8421"`: `(8,4)` cùng chẵn và `8 > 4` ⇒ `"4821"`. Bài 4: ứng viên có dạng `s[:i] + s[i+1] + s[i] + s[i+2:]`.
- **Mức 3 (sâu):** Bài 5(a): chỉ đổi điều kiện `a > b` thành `a < b`. Bài 5(b): duyệt `i`, mở rộng `j` đến hết đoạn cùng chẵn lẻ, `sorted(s[i:j+1])`, nối các đoạn đã sắp xếp. Có thể kiểm tra bằng BFS trên chuỗi ngắn như ở mục 15.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert ("43520" < "45320") is True
assert (int("8") % 2 == int("2") % 2) is True
assert "abc"[1] + "abc"[0] == "ba"
assert Solution().getSmallestString("21") == "21"
assert Solution().getSmallestString("31") == "13"

# Bài 2
assert Solution().getSmallestString("2864") == "2684"
assert Solution().getSmallestString("6572") == "6572"
assert Solution().getSmallestString("8421") == "4821"


# Bài 3
def get_smallest_todo_ans(s):
    for i in range(len(s) - 1):
        a, b = int(s[i]), int(s[i + 1])
        if a % 2 == b % 2 and a > b:
            return s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s


for s, expected in examples:
    assert get_smallest_todo_ans(s) == expected


# Bài 4
def smallest_bruteforce_todo_ans(s):
    candidates = [s]
    for i in range(len(s) - 1):
        if int(s[i]) % 2 == int(s[i + 1]) % 2:
            candidates.append(s[:i] + s[i + 1] + s[i] + s[i + 2:])
    return min(candidates)


rng = random.Random(19)
for _ in range(500):
    s = "".join(rng.choice("0123456789") for _ in range(rng.randint(2, 12)))
    assert smallest_bruteforce_todo_ans(s) == Solution().getSmallestString(s), s


# Bài 5
def largest_todo_ans(s):
    for i in range(len(s) - 1):
        a, b = int(s[i]), int(s[i + 1])
        if a % 2 == b % 2 and a < b:
            return s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s


def largest_bruteforce(s):
    candidates = [s]
    for i in range(len(s) - 1):
        if int(s[i]) % 2 == int(s[i + 1]) % 2:
            candidates.append(s[:i] + s[i + 1] + s[i] + s[i + 2:])
    return max(candidates)


def smallest_unlimited_todo_ans(s):
    out, i, n = [], 0, len(s)
    while i < n:
        j = i
        while j + 1 < n and int(s[j + 1]) % 2 == int(s[i]) % 2:
            j += 1
        out.append("".join(sorted(s[i:j + 1])))
        i = j + 1
    return "".join(out)


for _ in range(500):
    s = "".join(rng.choice("0123456789") for _ in range(rng.randint(2, 6)))
    assert largest_todo_ans(s) == largest_bruteforce(s), s
    assert smallest_unlimited_todo_ans(s) == smallest_unlimited_bfs(s), s
assert largest_todo_ans("45320") == "45320"
assert largest_todo_ans("1357") == "3157"
assert smallest_unlimited_todo_ans("9731") == "1379"
assert smallest_unlimited_todo_ans("45320") == "43502" == smallest_unlimited_bfs("45320")
print("Tất cả đáp án bài tập đều PASS")

Tất cả đáp án bài tập đều PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tham lam từ trái sang phải; đổi đúng một lần rồi dừng)

```python
class Solution:
    def getSmallestString(self, s: str) -> str:
        for i in range(len(s) - 1):
            a, b = int(s[i]), int(s[i + 1])
            if a % 2 == b % 2 and a > b:
                return s[:i] + s[i + 1] + s[i] + s[i + 2:]
        return s
```

**Checklist**

- [ ] Có kiểm tra **cùng chẵn lẻ** (`a % 2 == b % 2`) trước khi đổi?
- [ ] Điều kiện đổi là `a > b` **ngặt** (không phải `>=`)?
- [ ] Chọn cặp **đầu tiên** từ trái sang phải (không phải mức giảm lớn nhất)?
- [ ] Đổi **đúng một lần** rồi `return`/`break`?
- [ ] Không có cặp nào thì trả về chính `s`?
- [ ] Đã thử `"45320"` (`"43520"`), `"001"`, `"5391"`, `"5531"`, `"9731"`, và chuỗi 2 chữ số?